# 02 — Semantic Feature Mapping: Validasi Statistik

Tujuan: membuktikan apakah pasangan fitur kandidat (CIC-IDS2018 ↔ UNSW-NB15) benar-benar **sepadan secara kuantitatif**, bukan sekadar mirip nama. Karena kedua dataset diekstrak dengan alat berbeda (CICFlowMeter vs Argus/Bro), padanan wajib diverifikasi rentang/distribusi/satuan-nya.

**Penting soal skala CIC:** `cleaned_100.pkl` menyimpan fitur yang SUDAH di-`StandardScaler`. Untuk perbandingan yang adil dengan UNSW (raw), notebook ini **mengembalikan skala asli** CIC via `X_orig = X_scaled * scale_ + mean_`.

Output: `mapping_validation.csv` + `mapping_validation.json` (statistik per pasangan + verdict).

> Jalankan di SageMaker (butuh `cleaned_100.pkl`).

In [ ]:
# --- Bootstrap dependency ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json
import numpy as np
import pandas as pd

CIC_PKL   = '../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_CSV  = '../data/UNSW_NB15_testing-set.csv'   # 175.341 record (data latih menurut jumlah)
OUT_CSV   = '../mapping_validation.csv'
OUT_JSON  = '../mapping_validation.json'
print('CIC pkl :', os.path.exists(CIC_PKL))
print('UNSW    :', os.path.exists(UNSW_CSV))

In [ ]:
# --- Muat CIC-IDS2018 & kembalikan ke SKALA ASLI (un-scale) ---
with open(CIC_PKL, 'rb') as f:
    d = pickle.load(f)
cic_feats = list(d['feature_names'])
X = np.asarray(d['X'], dtype=float)
scaler = d.get('scaler', None)

if scaler is not None and hasattr(scaler, 'scale_') and hasattr(scaler, 'mean_'):
    X_orig = X * scaler.scale_ + scaler.mean_
    print('CIC: un-scaled memakai scaler.mean_/scale_')
else:
    X_orig = X
    print('CIC: scaler tidak tersedia -> pakai X apa adanya (hati-hati interpretasi)')

cic_df = pd.DataFrame(X_orig, columns=cic_feats)
print('CIC shape:', cic_df.shape)

In [ ]:
# --- Muat UNSW-NB15 (raw) ---
unsw_df = pd.read_csv(UNSW_CSV)
print('UNSW shape:', unsw_df.shape)

In [ ]:
# --- Pasangan kandidat (hipotesis dari dokumentasi) ---
# (nama CIC, nama UNSW, kategori hipotesis)
pairs = [
    ('Flow Duration',     'dur',    'kuat'),
    ('Tot Fwd Pkts',      'spkts',  'kuat'),
    ('Tot Bwd Pkts',      'dpkts',  'kuat'),
    ('TotLen Fwd Pkts',   'sbytes', 'kuat'),
    ('TotLen Bwd Pkts',   'dbytes', 'kuat'),
    ('Fwd Pkt Len Mean',  'smean',  'kuat'),
    ('Bwd Pkt Len Mean',  'dmean',  'kuat'),
    ('Init Fwd Win Byts', 'swin',   'kuat'),
    ('Init Bwd Win Byts', 'dwin',   'kuat'),
    ('Flow Byts/s',       'sload',  'kuat'),
    ('Bwd Pkts/s',        'dload',  'kuat'),
    ('Fwd IAT Mean',      'sinpkt', 'sedang'),
    ('Bwd IAT Mean',      'dinpkt', 'sedang'),
]

def stat(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    q = s.quantile([0.5, 0.99])
    return dict(min=float(s.min()), median=float(q.loc[0.5]), p99=float(q.loc[0.99]),
                max=float(s.max()), mean=float(s.mean()), std=float(s.std()))

rows = []
for cic_c, unsw_c, hyp in pairs:
    if cic_c not in cic_df.columns:
        rows.append(dict(cic=cic_c, unsw=unsw_c, hyp=hyp, note='CIC col MISSING')); continue
    if unsw_c not in unsw_df.columns:
        rows.append(dict(cic=cic_c, unsw=unsw_c, hyp=hyp, note='UNSW col MISSING')); continue
    cs, us = stat(cic_df[cic_c]), stat(unsw_df[unsw_c])
    # rasio median & p99 sebagai indikator kesetaraan skala
    def ratio(a, b):
        return float(a / b) if b not in (0, 0.0) else float('inf')
    rows.append(dict(
        cic=cic_c, unsw=unsw_c, hyp=hyp,
        cic_median=round(cs['median'], 4), unsw_median=round(us['median'], 4),
        cic_p99=round(cs['p99'], 2), unsw_p99=round(us['p99'], 2),
        cic_max=round(cs['max'], 2), unsw_max=round(us['max'], 2),
        ratio_median=round(ratio(cs['median'], us['median']), 3),
        ratio_p99=round(ratio(cs['p99'], us['p99']), 3),
        note=''))

res = pd.DataFrame(rows)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print(res.to_string(index=False))

In [ ]:
# --- Verdict otomatis berdasarkan kedekatan skala (median & p99) ---
# Aturan sederhana: jika rasio median & p99 berada di orde yang sama (0.1x..10x) -> 'aligned';
# jika berbeda 1-3 orde -> 'scale-mismatch (perlu konversi)'; >3 orde -> 'likely-different-feature'.
def verdict(r):
    if 'note' in r and r['note'] and 'MISSING' in str(r['note']):
        return 'missing'
    rat = r.get('ratio_p99', None)
    if rat is None or rat != rat or rat in (float('inf'),):
        return 'check-manual'
    a = abs(np.log10(rat)) if rat > 0 else 99
    if a <= 1:   return 'aligned'
    if a <= 3:   return 'scale-mismatch (perlu konversi/scaling)'
    return 'likely-different-feature'

res['verdict'] = res.apply(verdict, axis=1)
print(res[['cic','unsw','hyp','ratio_median','ratio_p99','verdict']].to_string(index=False))

res.to_csv(OUT_CSV, index=False)
res.to_json(OUT_JSON, orient='records', indent=2)
print('\nSaved:', OUT_CSV, '&', OUT_JSON)
print('\nCatatan: verdict ini indikator awal berbasis skala; konfirmasi akhir tetap perlu penalaran domain (satuan, definisi extractor).')